# PBMC ATAC–RNA aligned UMAP visualization

This notebook visualizes the **shared aligned latent representation** learned by scMRDR on PBMC.

PBMC does not contain the required cell-type labels in the original training data, so this notebook loads the separately generated Azimuth annotations from:

```text
/data5/zhangye/scMRDR/input/PBMC/raw_input/pbmc_azimuth_annotations.csv
```

and matches them to cells before plotting.

The modality-colored and cell-type-colored panels use the **same UMAP coordinates**.


## 1. Imports

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import warnings
import re

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from matplotlib.lines import Line2D

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets is not installed. Interactive controls will be skipped.")

sc.settings.verbosity = 2

print("scanpy:", sc.__version__)
print("ipywidgets available:", HAS_WIDGETS)


## 2. Settings

Azimuth L2 is the default annotation level. You can switch to L1 or L3.


In [ ]:
PROJECT_ROOT = Path("/data5/zhangye/scMRDR")

EXPERIMENT_NAME = "PBMC ATAC-RNA"
OUT_ROOT = PROJECT_ROOT / "output/PBMC/scMRDR_results"

PBMC_ANNOTATION_PATH = (
    PROJECT_ROOT / "input/PBMC/raw_input/pbmc_azimuth_annotations.csv"
)

POST_H5AD_NAME = "training_adata_post.h5ad"
LATENT_KEY = "latent_shared"
MODALITY_KEY = "modality"

ANNOTATION_ID_COL = "cell_id"
ANNOTATION_COLUMNS = [
    "predicted.celltype.l1",
    "predicted.celltype.l2",
    "predicted.celltype.l3",
]
DEFAULT_CELLTYPE_KEY = "predicted.celltype.l2"

N_NEIGHBORS = 30
MIN_DIST = 0.35
RANDOM_STATE = 1234
MAX_CELLS = None

POINT_SIZE = 5.0
POINT_ALPHA = 0.75
DPI = 300

print("Output root:", OUT_ROOT)
print("PBMC annotation:", PBMC_ANNOTATION_PATH)


## 3. PBMC annotation utilities

The external Azimuth table is matched by `cell_id`. If the training AnnData already contains a `cell_id` column, it is used; otherwise the notebook falls back to `obs_names`.


In [ ]:
_ANNOTATION_CACHE = None


def discover_ratios():
    if not OUT_ROOT.exists():
        return []

    return [
        d.name
        for d in sorted(OUT_ROOT.glob("single_*"))
        if d.is_dir() and (d / POST_H5AD_NAME).exists()
    ]


def canonicalize_cell_id(value):
    """Extract a standard 10x barcode from common concatenated IDs."""
    text = str(value)
    match = re.search(r"([ACGTN]{12,}-\d+)", text)
    return match.group(1) if match else text


def load_pbmc_annotations():
    global _ANNOTATION_CACHE

    if _ANNOTATION_CACHE is not None:
        return _ANNOTATION_CACHE

    if not PBMC_ANNOTATION_PATH.exists():
        raise FileNotFoundError(
            f"PBMC annotation file not found: {PBMC_ANNOTATION_PATH}"
        )

    ann = pd.read_csv(PBMC_ANNOTATION_PATH)

    required = [ANNOTATION_ID_COL] + ANNOTATION_COLUMNS
    missing = [c for c in required if c not in ann.columns]
    if missing:
        raise KeyError(
            f"Missing annotation columns: {missing}. "
            f"Available columns: {list(ann.columns)}"
        )

    ann = ann.copy()
    ann["_canonical_cell_id"] = ann[ANNOTATION_ID_COL].map(canonicalize_cell_id)

    if ann["_canonical_cell_id"].duplicated().any():
        raise ValueError("Duplicated canonical cell IDs were found in the Azimuth CSV.")

    ann = ann.set_index("_canonical_cell_id", drop=False)
    _ANNOTATION_CACHE = ann
    return ann


def attach_pbmc_annotations(obs):
    """Attach Azimuth L1/L2/L3 labels to the training AnnData observations."""
    obs = obs.copy()
    ann = load_pbmc_annotations()

    if "cell_id" in obs.columns:
        raw_ids = obs["cell_id"].astype(str).tolist()
        id_source = "obs['cell_id']"
    else:
        raw_ids = obs.index.astype(str).tolist()
        id_source = "obs_names"

    canonical_ids = pd.Series(
        [canonicalize_cell_id(x) for x in raw_ids],
        index=np.arange(len(obs)),
        dtype="object",
    )

    obs["_annotation_cell_id"] = canonical_ids.to_numpy()

    for col in ANNOTATION_COLUMNS:
        obs[col] = canonical_ids.map(ann[col]).to_numpy()

    matched = obs[DEFAULT_CELLTYPE_KEY].notna().to_numpy()

    report = {
        "id_source": id_source,
        "matched_rows": int(matched.sum()),
        "total_rows": int(len(obs)),
        "match_rate": float(matched.mean()) if len(obs) else np.nan,
    }

    return obs, report


def pretty_modality(value):
    value = str(value).strip().lower()
    mapping = {"rna": "RNA", "atac": "ATAC", "protein": "Protein"}
    return mapping.get(value, str(value))


def inspect_training_output(ratio_label):
    path = OUT_ROOT / ratio_label / POST_H5AD_NAME
    if not path.exists():
        raise FileNotFoundError(path)

    source = sc.read_h5ad(path)

    print("File:", path)
    print("Shape:", source.shape)
    print("obsm keys:", list(source.obsm.keys()))
    print("obs columns before annotation:", list(source.obs.columns))

    annotated_obs, report = attach_pbmc_annotations(source.obs)

    print("\nAnnotation match report:")
    for key, value in report.items():
        print(f"  {key}: {value}")

    for key in ANNOTATION_COLUMNS:
        print(f"\n{key}:")
        print(annotated_obs[key].value_counts(dropna=False).head(30))


def load_aligned_embedding(
    ratio_label,
    max_cells=MAX_CELLS,
    random_state=RANDOM_STATE,
    celltype_key=DEFAULT_CELLTYPE_KEY,
):
    path = OUT_ROOT / ratio_label / POST_H5AD_NAME
    if not path.exists():
        raise FileNotFoundError(
            f"Missing training output: {path}\n"
            "Run PBMC training first."
        )

    source = sc.read_h5ad(path)

    if LATENT_KEY not in source.obsm:
        raise KeyError(
            f"{LATENT_KEY!r} not found. Available obsm keys: {list(source.obsm.keys())}"
        )

    if MODALITY_KEY not in source.obs.columns:
        raise KeyError(
            f"{MODALITY_KEY!r} not found. Available obs columns: {list(source.obs.columns)}"
        )

    if celltype_key not in ANNOTATION_COLUMNS:
        raise KeyError(
            f"celltype_key must be one of {ANNOTATION_COLUMNS}, got {celltype_key!r}"
        )

    latent = np.asarray(source.obsm[LATENT_KEY], dtype=np.float32)
    obs, report = attach_pbmc_annotations(source.obs)

    print(
        f"[{ratio_label}] Azimuth annotation matched "
        f"{report['matched_rows']}/{report['total_rows']} rows "
        f"({report['match_rate']:.1%}) using {report['id_source']}."
    )

    if report["match_rate"] < 0.90:
        warnings.warn(
            "Less than 90% of PBMC observations matched the Azimuth CSV. "
            "Inspect the cell identifiers before interpreting the cell-type UMAP."
        )

    if max_cells is not None and len(obs) > max_cells:
        rng = np.random.default_rng(random_state)
        keep = np.sort(rng.choice(len(obs), size=max_cells, replace=False))
        latent = latent[keep]
        obs = obs.iloc[keep].copy()

    plot_adata = ad.AnnData(X=latent, obs=obs)

    plot_adata.obs["_modality_plot"] = (
        plot_adata.obs[MODALITY_KEY].astype(str).map(pretty_modality)
    )
    plot_adata.obs["_celltype_plot"] = (
        plot_adata.obs[celltype_key].fillna("Unknown").astype(str)
    )

    plot_adata.uns["ratio_label"] = ratio_label
    plot_adata.uns["celltype_key"] = celltype_key
    plot_adata.uns["source_file"] = str(path)

    return plot_adata, celltype_key


def compute_umap(
    plot_adata,
    n_neighbors=N_NEIGHBORS,
    min_dist=MIN_DIST,
    random_state=RANDOM_STATE,
):
    result = plot_adata.copy()

    sc.pp.neighbors(
        result,
        n_neighbors=n_neighbors,
        use_rep="X",
        random_state=random_state,
    )
    sc.tl.umap(
        result,
        min_dist=min_dist,
        random_state=random_state,
    )

    return result


## 4. Plotting functions

In [ ]:
MODALITY_PALETTE = {
    "RNA": "#4C78A8",
    "ATAC": "#E78AC3",
    "Protein": "#59A14F",
}


def make_celltype_palette(labels):
    labels = sorted(pd.unique(pd.Series(labels).astype(str)))
    base = list(sc.pl.palettes.default_102)
    palette = {label: base[i % len(base)] for i, label in enumerate(labels)}
    if "Unknown" in palette:
        palette["Unknown"] = "#BDBDBD"
    return palette


def _clean_axis(ax, title):
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def _scatter_categories(ax, coords, labels, palette, point_size, alpha):
    categories = sorted(pd.unique(labels))
    if "Unknown" in categories:
        categories = ["Unknown"] + [x for x in categories if x != "Unknown"]

    for label in categories:
        mask = labels == label
        ax.scatter(
            coords[mask, 0],
            coords[mask, 1],
            s=point_size,
            alpha=min(alpha, 0.35) if label == "Unknown" else alpha,
            linewidths=0,
            rasterized=True,
            label=label,
            c=[palette.get(label, "#BDBDBD")],
        )
    return categories


def plot_alignment_pair(
    adata_umap,
    save_dir=None,
    file_prefix=None,
    point_size=POINT_SIZE,
    alpha=POINT_ALPHA,
    dpi=DPI,
):
    """Plot modality and cell type using the same UMAP coordinates."""
    coords = np.asarray(adata_umap.obsm["X_umap"])
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.3), dpi=dpi)

    modality = adata_umap.obs["_modality_plot"].astype(str).to_numpy()
    _scatter_categories(
        axes[0], coords, modality, MODALITY_PALETTE, point_size, alpha
    )
    _clean_axis(axes[0], "Colored by modality")
    axes[0].legend(
        title="Modality",
        frameon=False,
        markerscale=2,
        loc="best",
    )

    celltype_key = adata_umap.uns.get("celltype_key")
    if celltype_key is not None and "_celltype_plot" in adata_umap.obs:
        celltype = adata_umap.obs["_celltype_plot"].astype(str).to_numpy()
        palette = make_celltype_palette(celltype)
        categories = _scatter_categories(
            axes[1], coords, celltype, palette, point_size, alpha
        )
        _clean_axis(axes[1], f"Colored by cell type ({celltype_key})")

        axes[1].legend(
            title="Cell type",
            frameon=False,
            fontsize=8,
            title_fontsize=9,
            markerscale=1.8,
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            ncol=1 if len(categories) <= 14 else 2,
        )
    else:
        _clean_axis(axes[1], "Cell type annotation unavailable")
        axes[1].text(
            0.5,
            0.5,
            "No usable cell-type annotation was found.",
            ha="center",
            va="center",
            transform=axes[1].transAxes,
        )

    ratio = adata_umap.uns.get("ratio_label", "")
    fig.suptitle(f"{EXPERIMENT_NAME} — {ratio}", fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()

    if save_dir is not None:
        save_dir = Path(save_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        if file_prefix is None:
            file_prefix = f"aligned_umap_{ratio}"

        png_path = save_dir / f"{file_prefix}.png"
        pdf_path = save_dir / f"{file_prefix}.pdf"
        fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
        fig.savefig(pdf_path, bbox_inches="tight")
        print("Saved:", png_path)
        print("Saved:", pdf_path)

    plt.show()
    return fig


def save_umap_coordinates(adata_umap, path):
    """Save UMAP coordinates and plotting labels."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    df = pd.DataFrame(
        {
            "UMAP1": adata_umap.obsm["X_umap"][:, 0],
            "UMAP2": adata_umap.obsm["X_umap"][:, 1],
            "modality": adata_umap.obs["_modality_plot"].astype(str).to_numpy(),
        },
        index=adata_umap.obs_names,
    )

    if "_celltype_plot" in adata_umap.obs:
        df["cell_type"] = adata_umap.obs["_celltype_plot"].astype(str).to_numpy()

    df.to_csv(path)
    print("Saved:", path)


## 5. Inspect one completed ratio

Run this first to confirm that the Azimuth CSV matches the PBMC cells.


In [ ]:
available_ratios = discover_ratios()
print("Available ratios:", available_ratios)

if available_ratios:
    inspect_training_output(available_ratios[0])
else:
    print("No completed ratio folders were found under:", OUT_ROOT)


## 6. Quick UMAP

Default: Azimuth L2. Change `CELLTYPE_KEY` to L1 or L3 if desired.


In [ ]:
CELLTYPE_KEY = "predicted.celltype.l2"

available_ratios = discover_ratios()

if available_ratios:
    SELECTED_RATIO = available_ratios[0]

    plot_adata, resolved_key = load_aligned_embedding(
        SELECTED_RATIO,
        max_cells=MAX_CELLS,
        celltype_key=CELLTYPE_KEY,
    )

    adata_umap = compute_umap(plot_adata)

    figure_dir = OUT_ROOT / "figures" / "alignment_umap"
    safe_key = CELLTYPE_KEY.replace(".", "_")

    plot_alignment_pair(
        adata_umap,
        save_dir=figure_dir,
        file_prefix=f"aligned_umap_{SELECTED_RATIO}_{safe_key}",
    )

    save_umap_coordinates(
        adata_umap,
        figure_dir / f"aligned_umap_{SELECTED_RATIO}_{safe_key}_coordinates.csv",
    )
else:
    print("No completed ratio folders were found.")


## 7. Interactive explorer

Switch ratio, Azimuth annotation level, and UMAP parameters interactively.


In [ ]:
if HAS_WIDGETS:
    ratios = discover_ratios()

    if not ratios:
        print("No completed ratio folders were found.")
    else:
        ratio_widget = widgets.Dropdown(
            options=ratios,
            value=ratios[0],
            description="Ratio:",
            style={"description_width": "90px"},
        )

        celltype_widget = widgets.Dropdown(
            options=[
                ("Azimuth L1", "predicted.celltype.l1"),
                ("Azimuth L2", "predicted.celltype.l2"),
                ("Azimuth L3", "predicted.celltype.l3"),
            ],
            value=DEFAULT_CELLTYPE_KEY,
            description="Cell type:",
            style={"description_width": "90px"},
        )

        neighbors_widget = widgets.IntSlider(
            value=N_NEIGHBORS,
            min=5,
            max=80,
            step=5,
            description="Neighbors:",
            continuous_update=False,
            style={"description_width": "90px"},
        )

        min_dist_widget = widgets.FloatSlider(
            value=MIN_DIST,
            min=0.0,
            max=0.95,
            step=0.05,
            description="Min dist:",
            continuous_update=False,
            style={"description_width": "90px"},
        )

        output_widget = widgets.Output()

        def redraw(*_):
            with output_widget:
                clear_output(wait=True)
                try:
                    plot_adata, _ = load_aligned_embedding(
                        ratio_widget.value,
                        celltype_key=celltype_widget.value,
                    )
                    adata_umap = compute_umap(
                        plot_adata,
                        n_neighbors=neighbors_widget.value,
                        min_dist=min_dist_widget.value,
                        random_state=RANDOM_STATE,
                    )
                    plot_alignment_pair(adata_umap, save_dir=None)
                except Exception as exc:
                    print(type(exc).__name__ + ":", exc)

        ratio_widget.observe(redraw, names="value")
        celltype_widget.observe(redraw, names="value")
        neighbors_widget.observe(redraw, names="value")
        min_dist_widget.observe(redraw, names="value")

        controls = widgets.VBox(
            [
                widgets.HBox([ratio_widget, celltype_widget]),
                widgets.HBox([neighbors_widget, min_dist_widget]),
            ]
        )

        display(controls, output_widget)
        redraw()
else:
    print("Install ipywidgets to enable this section: pip install ipywidgets")


## 8. Compare all ratios

Use one Azimuth level across all ratios for a consistent biological comparison.


In [ ]:
def plot_all_ratios(
    celltype_key=None,
    n_neighbors=N_NEIGHBORS,
    min_dist=MIN_DIST,
    max_cells=MAX_CELLS,
    n_cols=3,
    save=True,
):
    ratios = discover_ratios()
    if not ratios:
        raise RuntimeError(f"No completed ratios found under {OUT_ROOT}")

    records = []

    for ratio in ratios:
        print("Processing:", ratio)
        plot_adata, resolved_key = load_aligned_embedding(
            ratio,
            max_cells=max_cells,
            celltype_key=celltype_key,
        )
        adata_umap = compute_umap(
            plot_adata,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            random_state=RANDOM_STATE,
        )
        records.append((ratio, adata_umap, resolved_key))

    # --------------------------------------------------------
    # Modality panels
    # --------------------------------------------------------
    n_rows = math.ceil(len(records) / n_cols)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.8 * n_cols, 4.2 * n_rows),
        squeeze=False,
        dpi=DPI,
    )

    for ax, (ratio, adata_umap, _) in zip(axes.ravel(), records):
        coords = adata_umap.obsm["X_umap"]
        labels = adata_umap.obs["_modality_plot"].astype(str).to_numpy()
        _scatter_categories(
            ax, coords, labels, MODALITY_PALETTE, POINT_SIZE, POINT_ALPHA
        )
        _clean_axis(ax, ratio)

    for ax in axes.ravel()[len(records):]:
        ax.axis("off")

    modality_labels = sorted(
        {
            label
            for _, adata_umap, _ in records
            for label in adata_umap.obs["_modality_plot"].astype(str).unique()
        }
    )

    handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="",
            markerfacecolor=MODALITY_PALETTE.get(label, "#BDBDBD"),
            markeredgecolor="none",
            label=label,
        )
        for label in modality_labels
    ]

    fig.legend(
        handles=handles,
        title="Modality",
        frameon=False,
        loc="center right",
    )
    fig.suptitle(f"{EXPERIMENT_NAME}: aligned UMAP by modality", fontsize=14)
    fig.tight_layout(rect=[0, 0, 0.93, 0.96])

    save_dir = OUT_ROOT / "figures" / "alignment_umap"
    if save:
        save_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(
            save_dir / "all_ratios_umap_by_modality.png",
            dpi=DPI,
            bbox_inches="tight",
        )
        fig.savefig(
            save_dir / "all_ratios_umap_by_modality.pdf",
            bbox_inches="tight",
        )

    plt.show()

    # --------------------------------------------------------
    # Cell-type panels
    # --------------------------------------------------------
    valid = [
        (ratio, adata_umap, key)
        for ratio, adata_umap, key in records
        if key is not None and "_celltype_plot" in adata_umap.obs
    ]

    if not valid:
        print("No cell-type annotations were available.")
        return records

    all_celltypes = sorted(
        {
            label
            for _, adata_umap, _ in valid
            for label in adata_umap.obs["_celltype_plot"].astype(str).unique()
        }
    )
    palette = make_celltype_palette(all_celltypes)

    n_rows = math.ceil(len(valid) / n_cols)
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.8 * n_cols, 4.2 * n_rows),
        squeeze=False,
        dpi=DPI,
    )

    for ax, (ratio, adata_umap, _) in zip(axes.ravel(), valid):
        coords = adata_umap.obsm["X_umap"]
        labels = adata_umap.obs["_celltype_plot"].astype(str).to_numpy()
        _scatter_categories(
            ax, coords, labels, palette, POINT_SIZE, POINT_ALPHA
        )
        _clean_axis(ax, ratio)

    for ax in axes.ravel()[len(valid):]:
        ax.axis("off")

    handles = [
        Line2D(
            [0], [0],
            marker="o",
            linestyle="",
            markerfacecolor=palette[label],
            markeredgecolor="none",
            label=label,
        )
        for label in all_celltypes
    ]

    fig.legend(
        handles=handles,
        title="Cell type",
        frameon=False,
        fontsize=8,
        loc="center left",
        bbox_to_anchor=(0.93, 0.5),
        ncol=1 if len(all_celltypes) <= 15 else 2,
    )

    resolved_name = valid[0][2]
    fig.suptitle(
        f"{EXPERIMENT_NAME}: aligned UMAP by cell type ({resolved_name})",
        fontsize=14,
    )
    fig.tight_layout(rect=[0, 0, 0.88, 0.96])

    if save:
        safe_key = str(resolved_name).replace(".", "_").replace("/", "_")
        fig.savefig(
            save_dir / f"all_ratios_umap_by_celltype_{safe_key}.png",
            dpi=DPI,
            bbox_inches="tight",
        )
        fig.savefig(
            save_dir / f"all_ratios_umap_by_celltype_{safe_key}.pdf",
            bbox_inches="tight",
        )

    plt.show()
    return records


# Recommended:
# records = plot_all_ratios(celltype_key="predicted.celltype.l2")


## 9. Output

Figures are written to:

```text
output/PBMC/scMRDR_results/figures/alignment_umap/
```

PBMC is the only one of the three visualization notebooks that reads an external cell-type annotation CSV.
